## Extrair confrontos do HTML salvo
Arquivo salvo localmente com os confrontos da rodada.


In [7]:
from bs4 import BeautifulSoup
from pathlib import Path
import pandas as pd
import re

HTML_PATHS = [
    Path('confrontos_serie_A.html'),
    Path('confrontos_serie_B.html'),
    Path('confrontos_serie_C.html'),
    Path('confrontos_pontos_corridos.html'),
]

def extrair_confrontos(html_texto):
    soup = BeautifulSoup(html_texto, 'html.parser')
    cards = soup.select('div.card-confrontos-pontos-corridos')
    print(f'Cards encontrados: {len(cards)}')

    confrontos = []
    for card in cards:
        conf_el = card.select_one('.card-confrontos-pontos-corridos__confronto-texto')
        confronto_num = None
        if conf_el:
            m = re.search(r'(\d+)', conf_el.get_text(' ', strip=True))
            if m:
                confronto_num = int(m.group(1))

        items = card.select('.card-confrontos-pontos-corridos__item')
        if len(items) < 2:
            continue

        def nome_time(item):
            el = item.select_one('.card-confrontos-pontos-corridos__time')
            return el.get_text(' ', strip=True) if el else ''

        def id_time(item):
            a = None
            for _a in item.find_all('a', href=True):
                if '#!/time/' in _a.get('href',''):
                    a = _a
                    break
            if not a:
                return None
            href = a.get('href','')
            m = re.search(r'#!/time/(\d+)', href)
            return int(m.group(1)) if m else None

        mandante = nome_time(items[0])
        visitante = nome_time(items[1])
        id_a = id_time(items[0])
        id_b = id_time(items[1])

        if mandante and visitante:
            confrontos.append({
                'Rodada': 1,
                'Confronto': confronto_num,
                'Time A': mandante,
                'Time B': visitante,
                'ID A': id_a,
                'ID B': id_b,
            })
    return confrontos

for html_path in HTML_PATHS:
    if not html_path.exists():
        raise FileNotFoundError(f'Arquivo não encontrado: {html_path}')
    html = html_path.read_text(encoding='utf-8', errors='ignore')
    confrontos = extrair_confrontos(html)

    # calcula rodada por bloco de 10 confrontos
    for i, item in enumerate(confrontos):
        item['Rodada'] = (i // 10) + 1
    df_confrontos = pd.DataFrame(confrontos)
    df_confrontos = df_confrontos[['Rodada','Confronto','Time A','Time B','ID A','ID B']]
    display(df_confrontos)

    # salva no mesmo formato do CSV atual
    out = html_path.with_suffix('.csv')
    df_confrontos.to_csv(out, index=False)
    print(f'Salvo em: {out}')


Cards encontrados: 190


,Rodada,Confronto,Time A,Time B,ID A,ID B
0,1,1,FBC Colorado,Tatols Beants F.C,186283,212042
1,1,2,Mau Humor F.C.,Atlético Colorado 2021,19033717,44574146
2,1,3,Gremiomaniasm,Dom Camillo68,528730,20696550
3,1,4,MAFRA MARTINS FC,JV5 Tricolor Gaúcho,4911779,1747619
4,1,5,TORRESMO COM PINGA PRO26.2,Fedato Futebol Clube,49126346,18642587
...,...,...,...,...,...,...
185,19,6,Tatols Beants F.C,seralex,212042,29228373
186,19,7,TIGRE LEON,cartola scheuer17,3424598,3851966
187,19,8,Texas Club 2026,VASCO MARTINS FC,1273719,14696986
188,19,9,Rolo Compressor ZN,lsauer fc,18223508,44810918


Salvo em: confrontos_serie_A.csv
Cards encontrados: 190


,Rodada,Confronto,Time A,Time B,ID A,ID B
0,1,1,CAFÉ AMARGO PRO26.2,FBC Colorado,47547651,186283
1,1,2,JV5 Tricolor Gaúcho,PUXE FC,1747619,3447341
2,1,3,Sport Clube PAIM,TEAM LOPES 99,1148959,479510
3,1,4,TIGRE LEON,Pity10,3424598,48498051
4,1,5,cartola scheuer17,Tatols Beants F.C,3851966,212042
...,...,...,...,...,...,...
185,19,6,Paulo Virgili FC,Tatols Beants F.C,14124559,212042
186,19,7,lsauer fc,Dom Camillo68,44810918,20696550
187,19,8,Texas Club 2026,Gremiomaniasm,1273719,528730
188,19,9,Rolo Compressor ZN,Fedato Futebol Clube,18223508,18642587


Salvo em: confrontos_serie_B.csv
Cards encontrados: 190


,Rodada,Confronto,Time A,Time B,ID A,ID B
0,1,1,pura bucha/internacional,FBC Colorado,18661583,186283
1,1,2,Texas Club 2026,TIGRE LEON,1273719,3424598
2,1,3,Dom Camillo68,mercearia Estrela,20696550,25401606
3,1,4,TEAM LOPES 99,Mau Humor F.C.,479510,19033717
4,1,5,Tatols Beants F.C,Fedato Futebol Clube,212042,18642587
...,...,...,...,...,...,...
185,19,6,Fedato Futebol Clube,Rolo Compressor ZN,18642587,18223508
186,19,7,lsauer fc,Paulo Virgili FC,44810918,14124559
187,19,8,Gremiomaniasm,CAFÉ AMARGO PRO26.2,528730,47547651
188,19,9,cartola scheuer17,A Lenda Super Vasco F.c,3851966,117598


Salvo em: confrontos_serie_C.csv
Cards encontrados: 190


,Rodada,Confronto,Time A,Time B,ID A,ID B
0,1,1,FBC Colorado,GE Bebum,186283,16411206
1,1,2,GaúchoDaFronteira F.C,Pepe Leal FC,2371918,1326835
2,1,3,Grêmio_Campeão_LA_27,SC 100 Sono,47775950,14709358
3,1,4,Medonho´s F.C.,C R Juvenal,1867254,1488983
4,1,5,bugredasmissões,Arran Katoko FC,19209079,19833277
...,...,...,...,...,...,...
185,19,6,La Primeira Patada Es Nuestra,lsauer fc,32966,44810918
186,19,7,Texas Club 2026,GE Bebum,1273719,16411206
187,19,8,Pontaç0 F.C.,Esquadrão Gazembrino,20651178,2916559
188,19,9,GrioTeam,NHU PORÃ SAF.,14933455,4088673


Salvo em: confrontos_pontos_corridos.csv
